# GEO-Bench downstream prediction visualization

This notebook visualizes predictions from every completed frozen downstream task:

- **Classification:** m-bigearthnet, m-brick-kiln, m-so2sat, and m-eurosat.
- **Segmentation:** m-cashew-plant and m-SA-crop-type.

It does not rerun the encoder. It combines the cached test embeddings or dense features with the validation-selected `best_head.pth`. The displayed head is one validation-selected seed, not an ensemble or the five-seed aggregate reported in `results.json`.

## 1. Setup

Set the experiment and GEO-Bench roots below or with `GEO_MOE_EXPERIMENT_ROOT` and `GEO_BENCH_DIR`. The task lists default to all GEO-Bench tasks evaluated by this repository. Reduce the per-task example counts for a quicker local check.

In [ ]:
import json
import os
import sys
import textwrap
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch
import numpy as np
import torch

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from datasets.geobench import (
    CLASSIFICATION_DATASETS,
    SEGMENTATION_DATASETS,
    GeoBenchClassificationDataset,
    GeoBenchSegmentationDataset,
)
from utils.linear_probe import load_linear_probe_checkpoint
from utils.segmentation_probe import (
    load_segmentation_probe_checkpoint,
    prepare_dense_candidate,
)

experiment_root = Path(os.environ['MEOX_EXPERIMENT_ROOT'])
geobench_root = Path(os.environ['GEO_BENCH_DIR'])

classification_protocol = "geobench_classification_224"
classification_dataset_names = list(CLASSIFICATION_DATASETS)
classification_examples_per_task = 8
classification_selection = "mixed"  # random, mixed, or errors

segmentation_protocol = "geobench_segmentation_224"
segmentation_dataset_names = list(SEGMENTATION_DATASETS)
segmentation_examples_per_task = 8

random_seed = 42
device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

print(f"Experiment: {experiment_root}")
print(f"GEO-Bench:  {geobench_root}")
print(f"Device:     {device}")

## 2. Shared helpers

The extraction signature records whether evaluation used the complete test split or a seeded subset. Before plotting, `cached_sample` reconstructs the exact dataset row and asserts both the sample ID and ground-truth label against the cache. Therefore, a displayed image cannot silently inherit another cache row's label. RGB values are converted back from GEO-Bench's per-band z-score normalization and contrast-stretched jointly across B4/B3/B2 for display only.

In [ ]:
def load_json(path):
    with open(path, "r", encoding="utf-8") as handle:
        return json.load(handle)


def result_directory(protocol, dataset_name):
    path = experiment_root / "eval" / protocol / dataset_name
    required = [path / "results.json", path / "best_head.pth"]
    missing = [str(item) for item in required if not item.exists()]
    if missing:
        raise FileNotFoundError(f"Missing downstream artifacts: {missing}")
    return path


def optional_image_size(value):
    return None if value in {None, "native"} else int(value)


def subset_indices(dataset_length, signature):
    limit = signature.get("max_samples")
    if limit is None or dataset_length <= int(limit):
        return np.arange(dataset_length, dtype=np.int64)
    generator = torch.Generator().manual_seed(
        int(signature.get("sample_seed", 42))
    )
    return torch.randperm(
        dataset_length, generator=generator
    )[:int(limit)].numpy()


def class_names_from_dataset(dataset):
    label_type = dataset.task_specs.label_type
    names = getattr(label_type, "class_names", None)
    if names is None:
        names = getattr(label_type, "class_name", None)
    if names is None:
        names = [str(index) for index in range(dataset.num_classes)]
    names = [str(name) for name in names]
    if len(names) != dataset.num_classes:
        raise RuntimeError(
            f"Expected {dataset.num_classes} class names, got {len(names)}"
        )
    return names


def sentinel2_rgb(
    sample, dataset, lower_percentile=2, upper_percentile=98
):
    if "sentinel2" not in sample["raster_dict"]:
        raise ValueError(
            "RGB visualization requires Sentinel-2 in this dataset"
        )
    target_names = list(dataset.raster_band_names["sentinel2"])
    required = ("B4", "B3", "B2")
    if not all(name in target_names for name in required):
        raise ValueError(
            f"Sentinel-2 RGB bands are unavailable: {target_names}"
        )

    normalized = (
        sample["raster_dict"]["sentinel2"].detach().cpu().numpy()
    )
    validity = (
        sample["raster_valid_masks"]["sentinel2"]
        .detach().cpu().numpy()
    )
    source_by_target = {
        target: source
        for source, target in dataset.band_mapping["sentinel2"]
    }
    raw_channels = []
    valid_channels = []
    for target in required:
        channel_index = target_names.index(target)
        source = source_by_target[target]
        stats = dataset.dataset.band_stats[source]
        raw_channels.append(
            normalized[channel_index] * float(stats.std)
            + float(stats.mean)
        )
        valid_channels.append(validity[channel_index])

    rgb = np.stack(raw_channels, axis=-1)
    valid = np.stack(valid_channels, axis=-1).all(axis=-1)
    values = rgb[valid]
    if values.size == 0:
        return np.zeros_like(rgb, dtype=np.float32)
    low, high = np.percentile(
        values, [lower_percentile, upper_percentile]
    )
    if not np.isfinite(high - low) or high <= low:
        low, high = float(values.min()), float(values.max())
    scaled = np.clip(
        (rgb - low) / max(high - low, 1e-6), 0, 1
    )
    scaled[~valid] = 0
    return np.power(scaled, 0.85).astype(np.float32)


def cached_sample(
    dataset, ordered_indices, cached_ids, cached_label, cache_row
):
    dataset_index = int(ordered_indices[int(cache_row)])
    sample = dataset[dataset_index]
    cached_id = str(cached_ids[int(cache_row)])
    actual_id = str(sample["sample_id"])
    if actual_id != cached_id:
        raise RuntimeError(
            "Cache/dataset ID mismatch at row "
            f"{cache_row}: {cached_id!r} != {actual_id!r}"
        )

    actual_label = sample["label"].detach().cpu().numpy()
    expected_label = np.asarray(cached_label)
    if not np.array_equal(actual_label, expected_label):
        raise RuntimeError(
            "Cache/dataset label mismatch at row "
            f"{cache_row} for sample {actual_id!r}"
        )
    return sample


def wrapped_names(indices, names, width=48):
    values = [names[int(index)] for index in indices]
    joined = ", ".join(values) if values else "(none)"
    return textwrap.fill(joined, width=width)


def choose_classification_rows(
    labels, predictions, count, multilabel, mode, seed
):
    if mode not in {"random", "mixed", "errors"}:
        raise ValueError(
            "classification_selection must be random, mixed, or errors"
        )
    rng = np.random.default_rng(seed)
    count = min(int(count), len(labels))
    all_rows = np.arange(len(labels))
    if mode == "random":
        return np.sort(rng.choice(all_rows, size=count, replace=False))

    if multilabel:
        truth = labels.astype(bool)
        predicted = predictions.astype(bool)
        intersection = (truth & predicted).sum(axis=1)
        union = (truth | predicted).sum(axis=1)
        quality = np.divide(
            intersection,
            union,
            out=np.ones_like(intersection, dtype=float),
            where=union > 0,
        )
        candidates = np.flatnonzero(quality < 1)
        if mode == "errors" and len(candidates) >= count:
            return np.sort(
                rng.choice(candidates, size=count, replace=False)
            )
        order = np.argsort(quality)
        low_count = count // 2
        selected = np.concatenate(
            [order[:low_count], order[-(count - low_count):]]
        )
        return np.sort(np.unique(selected))

    correct = np.flatnonzero(predictions == labels)
    errors = np.flatnonzero(predictions != labels)
    if mode == "errors" and len(errors) >= count:
        return np.sort(rng.choice(errors, size=count, replace=False))
    error_count = min(len(errors), count // 2)
    correct_count = min(len(correct), count - error_count)
    selected = []
    if error_count:
        selected.extend(
            rng.choice(errors, size=error_count, replace=False)
        )
    if correct_count:
        selected.extend(
            rng.choice(correct, size=correct_count, replace=False)
        )
    remaining = np.setdiff1d(
        all_rows, np.asarray(selected, dtype=int)
    )
    if len(selected) < count:
        selected.extend(
            rng.choice(
                remaining, size=count - len(selected), replace=False
            )
        )
    return np.sort(np.asarray(selected, dtype=int))

## 3. Classification: all tasks

`mixed` is a diagnostic sample, not a representative random sample: for single-label tasks it deliberately displays roughly half errors and half correct predictions; for multilabel BigEarthNet it combines low- and high-Jaccard examples. Use `random` to inspect typical test examples or `errors` to stress-test mistakes.

For every displayed row, the notebook verifies `cached sample ID == live GEO-Bench sample ID` and `cached label == live GEO-Bench label`. Passing these checks rules out cache-order and class-index mismatches. It does **not** prove that the source scene-level label is semantically perfect: a EuroSAT tile can contain several land-cover types, and some official labels are visually ambiguous or noisy.

In [ ]:
def load_classification_task(dataset_name, seed):
    task_dir = result_directory(
        classification_protocol, dataset_name
    )
    manifest = load_json(task_dir / "results.json")
    cache_path = task_dir / "test_embeddings.npz"
    if not cache_path.exists():
        raise FileNotFoundError(cache_path)

    with np.load(cache_path, allow_pickle=False) as cache:
        embeddings = cache["embeddings"].copy()
        labels = cache["labels"].copy()
        sample_ids = cache["sample_ids"].astype(str)
        signature = json.loads(
            str(cache["extraction_signature"].item())
        )

    dataset = GeoBenchClassificationDataset(
        root_dir=geobench_root,
        dataset_name=dataset_name,
        split="test",
        model_band_names=manifest["model_band_names"],
        s1_modality=manifest.get("s1_modality", "sentinel1_asc"),
        partition_name=manifest.get("partition", "default"),
        input_image_size=optional_image_size(
            manifest["input_image_size"]
        ),
    )
    order = subset_indices(len(dataset), signature)
    cache_length = len(embeddings)
    if not (
        len(order) == cache_length == len(labels) == len(sample_ids)
    ):
        raise RuntimeError(
            "Classification cache arrays do not match the extraction "
            f"signature for {dataset_name}"
        )

    head, head_info = load_linear_probe_checkpoint(
        task_dir / manifest["head_checkpoint"], device=device
    )
    with torch.inference_mode():
        logits = head(
            torch.from_numpy(embeddings).float().to(device)
        ).cpu().numpy()

    multilabel = bool(head_info["multilabel"])
    if multilabel:
        probabilities = 1 / (
            1 + np.exp(-np.clip(logits, -80, 80))
        )
        predictions = probabilities >= 0.5
    else:
        shifted = logits - logits.max(axis=1, keepdims=True)
        exp_logits = np.exp(shifted)
        probabilities = exp_logits / exp_logits.sum(
            axis=1, keepdims=True
        )
        predictions = logits.argmax(axis=1)

    rows = choose_classification_rows(
        labels,
        predictions,
        classification_examples_per_task,
        multilabel,
        classification_selection,
        seed,
    )
    return {
        "name": dataset_name,
        "manifest": manifest,
        "dataset": dataset,
        "order": order,
        "labels": labels,
        "sample_ids": sample_ids,
        "probabilities": probabilities,
        "predictions": predictions,
        "multilabel": multilabel,
        "class_names": class_names_from_dataset(dataset),
        "rows": rows,
        "head_seed": head_info["seed"],
        "cache_length": cache_length,
    }

In [ ]:
def plot_classification_task(task):
    rows = task["rows"]
    figure, axes = plt.subplots(
        len(rows),
        2,
        figsize=(12, 3.3 * len(rows)),
        gridspec_kw={"width_ratios": [1, 1.8]},
        squeeze=False,
    )

    for row_axes, cache_row in zip(axes, rows):
        sample = cached_sample(
            task["dataset"],
            task["order"],
            task["sample_ids"],
            task["labels"][cache_row],
            cache_row,
        )
        row_axes[0].imshow(
            sentinel2_rgb(sample, task["dataset"])
        )
        row_axes[0].set_title(str(sample["sample_id"]), fontsize=10)
        row_axes[0].axis("off")

        if task["multilabel"]:
            gt_indices = np.flatnonzero(
                task["labels"][cache_row] > 0.5
            )
            pred_indices = np.flatnonzero(
                task["predictions"][cache_row]
            )
            top_indices = np.argsort(
                task["probabilities"][cache_row]
            )[-5:][::-1]
            top_text = ", ".join(
                f"{task['class_names'][index]} "
                f"{task['probabilities'][cache_row, index]:.2f}"
                for index in top_indices
            )
            intersection = len(set(gt_indices) & set(pred_indices))
            union = len(set(gt_indices) | set(pred_indices))
            agreement = intersection / union if union else 1.0
            text = (
                "Ground truth (cache/live verified):\n"
                f"{wrapped_names(gt_indices, task['class_names'])}\n\n"
                "Predicted at p >= 0.5:\n"
                f"{wrapped_names(pred_indices, task['class_names'])}\n\n"
                "Top probabilities:\n"
                f"{textwrap.fill(top_text, width=58)}\n\n"
                f"Label-set Jaccard: {agreement:.2f}"
            )
            color = "#235347" if agreement >= 0.5 else "#9b2c2c"
        else:
            gt_index = int(task["labels"][cache_row])
            pred_index = int(task["predictions"][cache_row])
            top_indices = np.argsort(
                task["probabilities"][cache_row]
            )[-3:][::-1]
            top_text = "\n".join(
                f"{task['class_names'][index]}: "
                f"{task['probabilities'][cache_row, index]:.3f}"
                for index in top_indices
            )
            is_correct = gt_index == pred_index
            text = (
                "Ground truth (cache/live verified): "
                f"{task['class_names'][gt_index]}\n\n"
                "Predicted: "
                f"{task['class_names'][pred_index]}\n\n"
                f"Top probabilities:\n{top_text}"
            )
            color = "#235347" if is_correct else "#9b2c2c"

        row_axes[1].axis("off")
        row_axes[1].text(
            0.02,
            0.95,
            text,
            transform=row_axes[1].transAxes,
            va="top",
            ha="left",
            fontsize=10,
            color=color,
            linespacing=1.35,
            bbox={
                "boxstyle": "round,pad=0.7",
                "facecolor": "#f7f4ed",
                "edgecolor": color,
            },
        )

    figure.suptitle(
        f"{task['name']} test predictions | "
        f"{task['manifest']['spatial_protocol']} | "
        f"selection={classification_selection}",
        fontsize=15,
        y=1.01,
    )
    figure.tight_layout()
    plt.show()
    print(
        f"{task['name']}: ID and label audit passed for "
        f"{len(rows)} displayed rows from {task['cache_length']} cached "
        f"test samples (head seed {task['head_seed']})."
    )


for task_index, dataset_name in enumerate(classification_dataset_names):
    classification_task = load_classification_task(
        dataset_name, random_seed + task_index
    )
    plot_classification_task(classification_task)
    del classification_task

## 4. Segmentation: all tasks

Only the selected cached samples pass through the saved UPerNet head. Ground truth and prediction use the same categorical palette, and the fourth panel marks incorrect pixels in red. Per-image mIoU is a qualitative diagnostic, not the dataset-level mIoU reported by the evaluation script. The same cache/live ID and label assertions run before every row is plotted.

In [ ]:
def load_segmentation_task(dataset_name, seed):
    task_dir = result_directory(segmentation_protocol, dataset_name)
    manifest = load_json(task_dir / "results.json")
    cache_path = task_dir / "test_dense_features.h5"
    if not cache_path.exists():
        raise FileNotFoundError(cache_path)

    with h5py.File(cache_path, "r") as cache:
        signature = json.loads(cache.attrs["extraction_signature"])
        cache_length = int(cache["features"].shape[0])
        sample_ids = cache["sample_ids"].asstr()[:]
        if not (
            cache_length == len(cache["labels"]) == len(sample_ids)
        ):
            raise RuntimeError(
                f"Inconsistent segmentation cache lengths for {dataset_name}"
            )

    dataset = GeoBenchSegmentationDataset(
        root_dir=geobench_root,
        dataset_name=dataset_name,
        split="test",
        model_band_names=manifest["model_band_names"],
        s1_modality=manifest.get("s1_modality", "sentinel1_asc"),
        partition_name=manifest.get("partition", "default"),
        input_image_size=optional_image_size(
            manifest["input_image_size"]
        ),
        label_image_size=optional_image_size(
            manifest["label_image_size"]
        ),
    )
    order = subset_indices(len(dataset), signature)
    if len(order) != cache_length:
        raise RuntimeError(
            "Segmentation cache length does not match its extraction "
            f"signature for {dataset_name}"
        )

    head, head_info = load_segmentation_probe_checkpoint(
        task_dir / manifest["head_checkpoint"], device=device
    )
    if head_info["candidate"].endswith("post_norm"):
        raise ValueError(
            "This notebook expects the frozen pre-norm segmentation protocol"
        )

    rng = np.random.default_rng(seed)
    rows = np.sort(
        rng.choice(
            np.arange(cache_length),
            size=min(segmentation_examples_per_task, cache_length),
            replace=False,
        )
    )
    samples = []
    with h5py.File(cache_path, "r") as cache, torch.inference_mode():
        for cache_row in rows:
            label = np.asarray(
                cache["labels"][cache_row], dtype=np.int64
            )
            sample = cached_sample(
                dataset,
                order,
                sample_ids,
                label,
                cache_row,
            )
            raw_features = torch.from_numpy(
                np.asarray(cache["features"][cache_row:cache_row + 1])
            ).to(device=device, dtype=torch.float32)
            candidate = prepare_dense_candidate(
                raw_features,
                head_info["candidate"],
                norm_weight=torch.ones(
                    raw_features.shape[2], device=device
                ),
                norm_bias=torch.zeros(
                    raw_features.shape[2], device=device
                ),
            )
            logits = head(candidate, target_size=label.shape)
            prediction = logits.argmax(dim=1)[0].cpu().numpy()
            samples.append((sample, label, prediction))

    return {
        "name": dataset_name,
        "manifest": manifest,
        "dataset": dataset,
        "class_names": class_names_from_dataset(dataset),
        "samples": samples,
        "rows": rows,
        "head_seed": head_info["seed"],
        "cache_length": cache_length,
    }

In [ ]:
def per_image_mean_iou(label, prediction, num_classes):
    values = []
    for class_index in range(num_classes):
        truth = label == class_index
        predicted = prediction == class_index
        union = np.logical_or(truth, predicted).sum()
        if union:
            intersection = np.logical_and(truth, predicted).sum()
            values.append(intersection / union)
    return float(np.mean(values)) if values else float("nan")


def plot_segmentation_task(task):
    dataset = task["dataset"]
    num_classes = dataset.num_classes
    base_palette = plt.get_cmap("tab20")(
        np.linspace(0, 1, max(num_classes, 2))
    )
    mask_cmap = ListedColormap(base_palette[:num_classes])
    mask_norm = BoundaryNorm(
        np.arange(-0.5, num_classes + 0.5), num_classes
    )

    figure, axes = plt.subplots(
        len(task["samples"]),
        4,
        figsize=(15, 3.8 * len(task["samples"])),
        squeeze=False,
    )
    for row_axes, (sample, label, prediction) in zip(
        axes, task["samples"]
    ):
        score = per_image_mean_iou(label, prediction, num_classes)
        row_axes[0].imshow(sentinel2_rgb(sample, dataset))
        row_axes[0].set_title(
            f"RGB | {sample['sample_id']}", fontsize=10
        )
        row_axes[1].imshow(
            label,
            cmap=mask_cmap,
            norm=mask_norm,
            interpolation="nearest",
        )
        row_axes[1].set_title("Ground truth (cache/live verified)")
        row_axes[2].imshow(
            prediction,
            cmap=mask_cmap,
            norm=mask_norm,
            interpolation="nearest",
        )
        row_axes[2].set_title(
            f"Prediction | image mIoU {score:.3f}"
        )
        row_axes[3].imshow(
            prediction != label,
            cmap=ListedColormap(["#f4f0e6", "#d1493f"]),
            vmin=0,
            vmax=1,
            interpolation="nearest",
        )
        row_axes[3].set_title(
            f"Errors | {(prediction != label).mean():.1%} pixels"
        )
        for axis in row_axes:
            axis.axis("off")

    legend_handles = [
        Patch(
            facecolor=base_palette[index],
            label=f"{index}: {name}",
        )
        for index, name in enumerate(task["class_names"])
    ]
    figure.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=min(5, num_classes),
        bbox_to_anchor=(0.5, -0.01),
        frameon=False,
    )
    figure.suptitle(
        f"{task['name']} test predictions | "
        f"{task['manifest']['spatial_protocol']}",
        fontsize=15,
        y=1.01,
    )
    figure.tight_layout(rect=(0, 0.05, 1, 1))
    plt.show()
    print(
        f"{task['name']}: ID and label audit passed for "
        f"{len(task['samples'])} displayed rows from "
        f"{task['cache_length']} cached test samples "
        f"(head seed {task['head_seed']})."
    )


for task_index, dataset_name in enumerate(segmentation_dataset_names):
    segmentation_task = load_segmentation_task(
        dataset_name, random_seed + 100 + task_index
    )
    plot_segmentation_task(segmentation_task)
    del segmentation_task

## 5. Reading the figures

- These examples are qualitative diagnostics, not replacements for complete test metrics in `results.json`.
- With `classification_selection = "mixed"`, mistakes are intentionally overrepresented. Do not infer the task error rate from these figures.
- A passing cache/live audit rules out row-order and label-decoding bugs for every displayed example. If an audited EuroSAT label still looks wrong, treat it as scene-label ambiguity or source annotation noise and inspect it under `random` sampling before making a dataset-wide claim.
- For segmentation, look for systematic boundary errors, missing small objects, confusion between adjacent classes, and spatially constant predictions.
- RGB is interpretive only. Its percentile stretch can make dark surfaces or atmospheric conditions look unusual; model inference uses all compatible bands with official GEO-Bench normalization.
- Keep the protocol name in exported figures. A 224 head must be paired with its 224 cache, and a model-input head with its own cache.